# Geospatial ML — Train on Colab & Save to Drive

This notebook trains / prepares all 3 model checkpoints needed for the API:
1. **ResNet-50 classifier** — land-cover classification (`best_model.pt`)
2. **Convolutional Autoencoder** — anomaly detection (`autoencoder_best.pt`)
3. **Mask R-CNN** — tree crown segmentation (`segmentation_best.pt`)

Checkpoints are saved to your Google Drive under `MyDrive/geospatial_checkpoints/`.

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/geospatial_checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_OUTPUT}')

## Step 2 — Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/iamvisheshsrivastava/geospatial
%cd geospatial
!git log --oneline -3

## Step 3 — Install dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    rasterio \
    wandb \
    boto3 \
    scikit-learn \
    numpy pandas matplotlib pillow \
    pydantic pydantic-settings \
    tqdm

import torch
print(f'PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 4 — Download & extract EuroSAT dataset (~90 MB)

Uses `requests` with SSL verification disabled — more reliable than `wget` because
we can validate the HTTP response and check the file is actually a ZIP before extracting.
Every valid ZIP starts with magic bytes `PK\x03\x04` — if the server returns an HTML
error page instead, we catch it immediately with a clear error.

Extraction searches the entire unpacked tree for class directories, so it works
regardless of how many nesting levels the ZIP uses internally.

In [ ]:
import os, zipfile, shutil, requests
from pathlib import Path
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]
EUROSAT_DIR = Path('data/eurosat')
EUROSAT_DIR.mkdir(parents=True, exist_ok=True)
zip_path = EUROSAT_DIR / 'EuroSAT.zip'

URL = 'https://madm.dfki.de/files/sentinel/EuroSAT.zip'
print(f'Downloading EuroSAT from {URL} ...')
resp = requests.get(URL, verify=False, stream=True, timeout=120)
resp.raise_for_status()

total = int(resp.headers.get('content-length', 0))
downloaded = 0
with open(zip_path, 'wb') as f:
    for chunk in resp.iter_content(chunk_size=65536):
        f.write(chunk)
        downloaded += len(chunk)
        if total:
            pct = downloaded / total * 100
            print(f'\r  {downloaded // 1_048_576} / {total // 1_048_576} MB  ({pct:.1f}%)', end='')
print(f'\nDownload complete. File size: {zip_path.stat().st_size / 1_048_576:.1f} MB')

with open(zip_path, 'rb') as f:
    magic = f.read(4)
if magic != b'PK\x03\x04':
    raise RuntimeError(
        f'Downloaded file is NOT a ZIP (first 4 bytes: {magic}). '
        'The server likely returned an HTML error page.'
    )
print('ZIP magic bytes OK.')

print('\nZIP contents (first 20 entries):')
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist()[:20]:
        print(f'  {name}')

tmp_dir = EUROSAT_DIR / '_tmp_extract'
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir()
print('\nExtracting...')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(tmp_dir)

found = {}
for dirpath, dirnames, _ in os.walk(tmp_dir):
    for d in list(dirnames):
        if d in CLASSES and d not in found:
            found[d] = Path(dirpath) / d

print(f'Found {len(found)}/10 class directories.')
for cls, src in found.items():
    dest = EUROSAT_DIR / cls
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(src), str(dest))

shutil.rmtree(tmp_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)

print('\nDataset verification:')
all_ok = True
for cls in CLASSES:
    d = EUROSAT_DIR / cls
    count = len(list(d.glob('*'))) if d.exists() else 0
    status = 'OK' if count > 0 else 'MISSING'
    if count == 0:
        all_ok = False
    print(f'  {status:7s}  {cls} ({count} images)')

if not all_ok:
    raise RuntimeError('Some class directories are missing — check ZIP contents above.')
print('\nAll 10 classes ready.')

## Step 5 — Train the ResNet-50 classifier

~10 minutes on T4 GPU. Fine-tunes a ResNet-50 (pretrained on ImageNet) on EuroSAT
to classify satellite patches into 10 land-cover types. Saves the best checkpoint
automatically based on validation F1 score.

In [ ]:
!python -m src.train \
    --data-root data/eurosat \
    --epochs 10 \
    --batch-size 64 \
    --learning-rate 3e-4 \
    --num-workers 2 \
    --checkpoint-dir checkpoints \
    --wandb-mode disabled

## Step 6 — Train the Anomaly Detector (Autoencoder)

~10 minutes on T4 GPU. Trains a convolutional autoencoder on Forest patches only.
The model learns what healthy forest looks like — at inference time, patches with
high reconstruction error (deforestation, fire, industrial intrusion) are flagged.

In [ ]:
!python -m src.anomaly \
    --data-root data/eurosat \
    --normal-classes Forest \
    --epochs 30 \
    --wandb-mode disabled

## Step 6b — Save Mask R-CNN for tree crown segmentation

Downloads a Mask R-CNN pretrained on COCO (80 object classes) from torchvision
and saves it as `segmentation_best.pt`. This gives the `/segment` endpoint a real
working model — it detects and segments objects in aerial imagery out of the box.
No custom training data needed.

In [ ]:
import torch, torchvision
from pathlib import Path

Path('checkpoints').mkdir(exist_ok=True)
print('Downloading pretrained Mask R-CNN weights...')
model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='DEFAULT')
torch.save(model.state_dict(), 'checkpoints/segmentation_best.pt')
size_mb = Path('checkpoints/segmentation_best.pt').stat().st_size / 1_048_576
print(f'Saved checkpoints/segmentation_best.pt ({size_mb:.1f} MB)')

## Step 7 — Verify all checkpoints were created

In [ ]:
import os
all_good = True
for f in ['checkpoints/best_model.pt', 'checkpoints/autoencoder_best.pt', 'checkpoints/segmentation_best.pt']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1_048_576
        print(f'  OK       {f}  ({size_mb:.1f} MB)')
    else:
        print(f'  MISSING  {f}')
        all_good = False

print('\nAll 3 checkpoints ready — proceed to Step 8.' if all_good else '\nSome checkpoints missing — check steps above.')

## Step 8 — Copy all checkpoints to Google Drive

Colab sessions are temporary — everything is deleted when the session ends.
This copies all 3 checkpoint files to your Google Drive so you can download them.

In [ ]:
import shutil, os

files_to_save = [
    'checkpoints/best_model.pt',
    'checkpoints/autoencoder_best.pt',
    'checkpoints/segmentation_best.pt',
    'checkpoints/training_history.json',
]

for src in files_to_save:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(src))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1_048_576
        print(f'  Saved  {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'  SKIPPED (not found): {src}')

print('\nDone! Download these files from Google Drive:')
print('  best_model.pt        — ResNet-50 classifier   → /predict endpoint')
print('  autoencoder_best.pt  — Anomaly detector       → /anomaly endpoint')
print('  segmentation_best.pt — Mask R-CNN             → /segment endpoint')

## Step 9 — Quick sanity check (optional)

Loads the trained classifier and runs a prediction on a random EuroSAT image.
If this prints a class name and confidence score, the model is working correctly.

In [ ]:
import torch
from pathlib import Path
from src.models.resnet import build_resnet50_classifier
from src.data.preprocessing import preprocess_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('checkpoints/best_model.pt', map_location=device)
class_names = ckpt['class_names']
model = build_resnet50_classifier(num_classes=len(class_names), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

sample = next(Path('data/eurosat').glob('*/*.jpg'))
tensor = preprocess_image(sample, 224).unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.softmax(model(tensor), dim=1).squeeze()

conf, idx = probs.max(0)
print(f'Image     : {sample}')
print(f'Predicted : {class_names[idx]} ({conf:.1%} confidence)')
print(f'Val F1    : {ckpt["metrics"]["macro_f1"]:.4f}')